In [1]:
import os
from dotenv import load_dotenv

# API 키는 노트북에 직접 공개하지 말고 환경변수로 설정하는 것을 권장합니다.
# 예: 운영체제나 VS Code의 환경변수에 OPENAI_API_KEY를 미리 설정합니다.
# 상위 폴더의 .env 파일에서 읽어오기

load_dotenv()


if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY 환경변수를 설정한 후 실행하세요.")


In [2]:
import openai
print("openai version:", openai.__version__)


openai version: 3.6.0


In [3]:
import json
import pandas as pd
import urllib.request
from openai import OpenAI

client = OpenAI()


In [4]:
urllib.request.urlretrieve('https://raw.githubusercontent.com/MrBananaHuman/CounselGPT/main/total_kor_multiturn_counsel_bot.jsonl', 'total_kor_multiturn_counsel_bot.jsonl')

with open('total_kor_multiturn_counsel_bot.jsonl', 'r', encoding='utf-8') as file:
    original_jsonl_data = [json.loads(line) for line in file]
print('데이터의 개수:', len(original_jsonl_data))

데이터의 개수: 8731


In [5]:
original_jsonl_data = original_jsonl_data[:200]
print('데이터의 개수:', len(original_jsonl_data))

데이터의 개수: 200


In [6]:
print(original_jsonl_data[0])

[{'speaker': '상담사', 'utterance': '안녕하세요. 상담사입니다. 무엇이 불편하시나요?'}, {'speaker': '내담자', 'utterance': '내가 약간 중2병 같은 걸 증상을 보이고 있어요.'}, {'speaker': '상담사', 'utterance': '중2병 증상이라니, 어떤 증상이신 건가요?'}, {'speaker': '내담자', 'utterance': '그러니까 공부하기 싫어하고, 공격적이고, 좀 무례하게 말하고 싶은 게 많아져서 그런 거예요.'}, {'speaker': '상담사', 'utterance': '그런 증상이 있으니까 힘드시겠죠. 중2병 같은 것이라고 생각하시는 이유는 무엇인가요?'}, {'speaker': '내담자', 'utterance': '막 공부 안하고 이것저것 들먹이고 하고 싶은 게 너무 많아서 그런 거 같아요.'}, {'speaker': '상담사', 'utterance': '그런 것도 어쩔 수 없이 찾아오는 시기가 있으니까 무리하지 않도록 해야겠죠. 대학교를 가면서 나아질 것 같았는데, 오히려 더 심해진 것 같다고 하셨죠. 그 원인이 무엇인가요?'}, {'speaker': '내담자', 'utterance': '그걸 제가 잘 몰라서 그런 것 같아요. 그냥 더 심해졌다고 느꼈어요.'}, {'speaker': '상담사', 'utterance': '대학교 생활이 신나고 재밌으신 건 어떤 점이 있나요?'}, {'speaker': '내담자', 'utterance': '학과가 정말 좋아서 즐겁게 수업을 듣고 있어요. 학우들도 좋고 괜찮은 친구들도 많이 만나서 그런 것 같아요.'}, {'speaker': '상담사', 'utterance': '즐거운 일도 많이 있으면서 고민거리도 있는 것 같군요. 가사나 소설을 쓰시면서 마음을 풀기도 하신다고 하셨는데, 언제부터 그 습관이 생겨난 건가요?'}, {'speaker': '내담자', 'utterance': '좋은 질문이에요. 좀 자세히 말씀드릴게요. 학교에서 어려운 일이 

In [7]:
speaker_dict = {'내담자': 'user', '상담사': 'assistant'}

def transform_to_new_format(original_data, speaker_dict):
    transformed_data = []
    for conversation in original_data:
        current_conversation = {"messages": [{"role": "system", "content": "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 성심성의껏 상담해주세요"}]}
        for item in conversation:
            current_conversation["messages"].append({
                "role": speaker_dict[item["speaker"]],
                "content": item["utterance"]
            })
        if current_conversation['messages'][-1]['role'] == 'user':
            current_conversation['messages'] = current_conversation['messages'][:-1]
        transformed_data.append(current_conversation)

    return transformed_data

result = transform_to_new_format(original_jsonl_data, speaker_dict)
print('데이터의 개수:', len(result))

데이터의 개수: 200


In [8]:
print(result[0])

{'messages': [{'role': 'system', 'content': '당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 성심성의껏 상담해주세요'}, {'role': 'assistant', 'content': '안녕하세요. 상담사입니다. 무엇이 불편하시나요?'}, {'role': 'user', 'content': '내가 약간 중2병 같은 걸 증상을 보이고 있어요.'}, {'role': 'assistant', 'content': '중2병 증상이라니, 어떤 증상이신 건가요?'}, {'role': 'user', 'content': '그러니까 공부하기 싫어하고, 공격적이고, 좀 무례하게 말하고 싶은 게 많아져서 그런 거예요.'}, {'role': 'assistant', 'content': '그런 증상이 있으니까 힘드시겠죠. 중2병 같은 것이라고 생각하시는 이유는 무엇인가요?'}, {'role': 'user', 'content': '막 공부 안하고 이것저것 들먹이고 하고 싶은 게 너무 많아서 그런 거 같아요.'}, {'role': 'assistant', 'content': '그런 것도 어쩔 수 없이 찾아오는 시기가 있으니까 무리하지 않도록 해야겠죠. 대학교를 가면서 나아질 것 같았는데, 오히려 더 심해진 것 같다고 하셨죠. 그 원인이 무엇인가요?'}, {'role': 'user', 'content': '그걸 제가 잘 몰라서 그런 것 같아요. 그냥 더 심해졌다고 느꼈어요.'}, {'role': 'assistant', 'content': '대학교 생활이 신나고 재밌으신 건 어떤 점이 있나요?'}, {'role': 'user', 'content': '학과가 정말 좋아서 즐겁게 수업을 듣고 있어요. 학우들도 좋고 괜찮은 친구들도 많이 만나서 그런 것 같아요.'}, {'role': 'assistant', 'content': '즐거운 일도 많이 있으면서 고민거리도 있는 것 같군요. 가사나 소설을 쓰시면서 마음을 풀기도 하신다고 하셨는데, 언제부터

In [9]:
def save_jsonl_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        for item in data:
            json.dump(item, file, ensure_ascii=False)
            file.write('\n')

save_jsonl_file(result, 'messages.jsonl')

In [10]:
with open("messages.jsonl", "rb") as file:
    uploaded_file = client.files.create(
        file=file,
        purpose="fine-tune"
    )

print("uploaded file:", uploaded_file.id)


uploaded file: file-BzVQfGTNcq9U1PtEJZRDFV


In [11]:
job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.id,
    model="gpt-4o-mini-2024-07-18"
)

print("fine-tuning job:", job.id)


PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

In [ ]:
job_status = client.fine_tuning.jobs.retrieve(job.id)
print("status:", job_status.status)


In [ ]:
job_status = client.fine_tuning.jobs.retrieve(job.id)
print("status:", job_status.status)
if job_status.status == "succeeded":
    print("fine-tuned model:", job_status.fine_tuned_model)


In [ ]:
job_status = client.fine_tuning.jobs.retrieve(job.id)

if job_status.status != "succeeded":
    raise RuntimeError(f"파인튜닝이 아직 완료되지 않았습니다: {job_status.status}")

fine_tuned_model = job_status.fine_tuned_model
print(fine_tuned_model)


In [ ]:
def respone_by_chatgpt(model, messages):
    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    print(completion.choices[0].message.content)

model = 'gpt-4o-mini'
system_prompt = "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 상담하면서 여러 가지 방법을 알려주세요."
user_input = "요즘 혼자인 것 같아서 외로워요"

messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

respone_by_chatgpt(model, messages)

In [ ]:
model = fine_tuned_model
respone_by_chatgpt(model, messages)


In [ ]:
user_first_input = "요즘 혼자인 것 같아서 외로워요"
assistant_answer = "혼자 있는 것 같아서 외로움을 느끼신다는 건, 친구나 가족과의 소통 부족으로 인해 자기감정을 충족시키지 못하고 있다는 것을 의미합니다. 내담자님이 외로움을 느끼는 이유에 대해 좀 더 말씀해주실 수 있을까요?"
user_second_input = "저도 뭐가 문제인지 모르겠어요"

messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_first_input},
    {"role": "assistant", "content": assistant_answer},
    {"role": "user", "content": user_second_input}
]
model = fine_tuned_model
respone_by_chatgpt(model, messages)


In [ ]:
import gradio as gr

def predict(user_input, history):
    history = history + [{"role": "user", "content": user_input}]

    gpt_response = client.chat.completions.create(
        model=fine_tuned_model,
        messages=history
    )

    response = gpt_response.choices[0].message.content
    history = history + [{"role": "assistant", "content": response}]
    return history, history


with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="ChatBot", type="messages")

    state = gr.State([{
        "role": "system",
        "content": "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 성심성의껏 상담해주세요."
    }])

    with gr.Row():
        txt = gr.Textbox(show_label=False,
                         placeholder="심리 상담 챗봇에게 심리 상담을 받아보세요.")

    txt.submit(predict, [txt, state], [chatbot, state])

demo.launch(debug=True, share=True)
